---
short_title: "CSV von Hand einlesen"
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# CSV manuell einlesen

In diesem Abschnitt zerlegen wir eine gelesene CSV-Datei „von Hand“ in ihre
Bestandteile. Fachlich heißt das **parsen** (to parse: analysieren und
zerlegen).

Wir beginnen mit einem einfachen Beispiel und sehen dann, wo dieses Vorgehen an
seine Grenzen stößt.

## Ein einfaches Beispiel

Eine {term}`CSV`-Datei besteht aus Zeilen. Die erste Zeile ist meist die **Kopfzeile**
(Header) mit den Spaltennamen, alle weiteren Zeilen sind **Datensätze**.

Betrachten wir eine kleine, „harmlose“ CSV-Datei:

```csv
isbn,title,year
9780141185422,Ulysses,1922
9783596196036,Der Prozess,1925
```

In [1]:
kleine_csv = "isbn,title,year\n9780141185422,Ulysses,1922\n9783596196036,Der Prozess,1925"

# split("\n") zerlegt den Text in einzelne Zeilen.
zeilen = kleine_csv.split("\n")
print(zeilen)

['isbn,title,year', '9780141185422,Ulysses,1922', '9783596196036,Der Prozess,1925']


In [2]:
# Header abtrennen: erste Zeile, am Komma zerlegt.
header = zeilen[0].split(",")
print("Header:", header)

# Die restlichen Zeilen als Liste von Listen aufbauen.
daten = []
for zeile in zeilen[1:]:
    daten.append(zeile.split(","))

print("Datensätze:", daten)

Header: ['isbn', 'title', 'year']
Datensätze: [['9780141185422', 'Ulysses', '1922'], ['9783596196036', 'Der Prozess', '1925']]


## Zerlegen mit `split()` und `strip()`

`split(",")` zerlegt einen String an jedem Komma. `strip()` entfernt Leerzeichen
und den Zeilenumbruch (`\n`) am Anfang und Ende.

In echten Dateien stehen oft zusätzliche Leerzeichen um die Werte. Diese
entfernen wir mit `strip()`.

In [3]:
datei = "../assets/data/books_powerlaw_dataset.csv"

with open(datei, encoding="utf-8") as f:
    zeilen = f.readlines()

# Header: Zeilenumbruch entfernen und am Komma zerlegen.
header = zeilen[0].strip().split(",")
print("Header:", header)

Header: ['isbn', 'author', 'year', 'title', 'sales_year', 'sales']


In [4]:
daten = []
for zeile in zeilen[1:]:
    # strip() entfernt den Zeilenumbruch am Ende,
    # split(",") zerlegt in die einzelnen Felder.
    felder = zeile.strip().split(",")
    daten.append(felder)

print("Anzahl Datensätze:", len(daten))
print("Erster Datensatz:", daten[0])

Anzahl Datensätze: 180
Erster Datensatz: ['9780241951446', '"Murakami', ' Haruki"', '1984', 'Norwegian Wood', '2020', '61']


## Ein Blick auf die Feldanzahl

Wenn unsere Zerlegung korrekt ist, müsste jeder Datensatz genauso viele Felder
haben wie der Header. Prüfen wir das nach.

In [5]:
anzahl_spalten = len(header)
print("Erwartete Feldanzahl:", anzahl_spalten)

for datensatz in daten:
    if len(datensatz) != anzahl_spalten:
        print("Abweichung gefunden:", datensatz)
        break

Erwartete Feldanzahl: 6
Abweichung gefunden: ['9780241951446', '"Murakami', ' Haruki"', '1984', 'Norwegian Wood', '2020', '61']


In [6]:
fehlerhafte = [d for d in daten if len(d) != len(header)]
print("Datensätze mit falscher Feldanzahl:", len(fehlerhafte), "von", len(daten))

Datensätze mit falscher Feldanzahl: 180 von 180


## Wo liegen die Grenzen des manuellen Parsens?

Im Datensatz stehen die Autor\*innennamen in doppelten Anführungszeichen, weil
sie ein Komma enthalten:

```csv
9780241951446,"Murakami, Haruki",1984,Norwegian Wood,2020,61
```

`split(",")` „sieht“ dieses Komma nicht und zerlegt den Namen in zwei Felder.
Aus sechs Spalten werden so sieben. Genau deshalb schlägt unsere manuelle
Zerlegung bei fast allen Zeilen fehl.

Weitere Fälle, die `split(",")` nicht bewältigt:

- Kommas innerhalb von Anführungszeichen (unser Fall),
- doppelte Anführungszeichen als **Escaping** innerhalb eines Feldes,
- Felder, die selbst einen Zeilenumbruch enthalten,
- ein anderes Trennzeichen wie `;` oder ein Tabulator,
- Leerzeilen oder Kommentarzeilen.

```{warning}
Regeln wie „zähle Anführungszeichen“ oder „überspringe Kommas zwischen Quotes“
selbst zu programmieren, ist fehleranfällig. Für CSV gibt es deshalb das
Standardmodul `csv`.
```

In [7]:
# Ein naiver Reparaturversuch: Anführungszeichen vor dem Zerlegen entfernen.
# Das ändert nichts an der Anzahl der Kommas und damit auch nicht
# an der (falschen) Feldanzahl.
zeile = zeilen[1].strip()
print(zeile.replace('"', "").split(","))

['9780241951446', 'Murakami', ' Haruki', '1984', 'Norwegian Wood', '2020', '61']


## Zwischenergebnis

Wir können eine Datei einlesen, den Header abtrennen und eine Liste von Listen
aufbauen. Das funktioniert für einfache CSV-Dateien, aber nicht zuverlässig für
komplexere.

Im nächsten Kapitel nutzen wir deshalb das Standardmodul `csv`, das diese Fälle
korrekt behandelt.

### Ausblick: Liste von Listen als Wörterbuch

Häufig ist es bequemer, statt über einen Zahlenindex (`felder[3]`) über einen
Spaltennamen (`buch["title"]`) zuzugreifen. Dazu kombiniert man `dict()` mit
`zip()`. Für unser kleines Beispiel funktioniert das gut:

```python
header = ["isbn", "title", "year"]
zeile = ["9780141185422", "Ulysses", "1922"]

buch = dict(zip(header, zeile))
print(buch)
# {'isbn': '9780141185422', 'title': 'Ulysses', 'year': '1922'}
```

Das setzt aber voraus, dass die Feldanzahl stimmt. Für echte Dateien erledigt
diesen Schritt `csv.DictReader` automatisch.